<a href="https://colab.research.google.com/github/stefkong1982/netology.ru/blob/Master/%D0%92%D1%81%D0%B5_%D0%92%D0%B7%D0%B0%D0%B8%D0%BC_%D0%9E%D0%B3%D1%80_100_ml_ozon_recsys_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# OZON RecSys Baseline - Рекомендательная система для категории Apparel
Этот ноутбук содержит базовое решение для задачи предсказания следующей покупки пользователя в категории одежды, обуви и аксессуаров.

## Задача
- Предсказать топ-100 товаров для каждого пользователя из тестовой выборки
- Метрика оценки: NDCG@100
- Данные: ~38GB в формате parquet, 1.6B взаимодействий, 19M заказов


In [15]:
# === 1. Монтируем Google Drive, задаём пути к данным (структура Colab/Яндекс) ===
from google.colab import drive
drive.mount('/content/drive')

# Шаг 1. Импорты и пути (Colab/локально, без лишних библиотек)
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from collections import defaultdict, Counter, deque # Добавлены для подсчета популярности
import glob

# Пути к данным (замените на свои)
ORDERS_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_orders_data/final_apparel_orders_data_07'
ORDERS_PATH2 = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/raw/ml_ozon_recsys_train_final_apparel_orders_data'
# НОВЫЙ ПУТЬ: Добавляем путь к данным взаимодействий за нужный период
TRACKER_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/extra_ml_ozon_recsys_train/archive_extra_tracker_data/final_apparel_tracker_data_08'
TEST_PATH = '/content/drive/MyDrive/Colab Notebooks/e_cup_2025_project/data/2_raw/ml_ozon_recsys_test'

# Шаг 2. Загрузка всех заказов (train)
def load_orders():
    print("Загружаем тренировочные данные заказов...")
    orders = []
    for path in [ORDERS_PATH, ORDERS_PATH2]:
        for f in Path(path).rglob('*.parquet'):
            orders.append(pd.read_parquet(f))
    df = pd.concat(orders, ignore_index=True)
    # Обогащаем created_date, если нужно
    if 'created_date' in df.columns and df['created_date'].isna().sum():
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_date'].fillna(df['created_timestamp'].dt.date)
        df['created_date'] = pd.to_datetime(df['created_date'])
    elif 'created_date' not in df.columns and 'created_timestamp' in df.columns:
        # Если created_date вообще отсутствует, создаем её
        df['created_timestamp'] = pd.to_datetime(df['created_timestamp'])
        df['created_date'] = df['created_timestamp'].dt.date
        df['created_date'] = pd.to_datetime(df['created_date'])
    print(f"Загружено заказов: {len(df):,}")
    return df

orders_df = load_orders()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Загружаем тренировочные данные заказов...
Загружено заказов: 20,362,338


In [16]:
print("=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(orders_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in orders_df.columns:
    print(f"  • {col}: {orders_df[col].dtype}")

=== 📘 ТРЕНИРОВОЧНЫЕ ДАННЫЕ ЗАКАЗОВ (orders_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|    |   item_id |   user_id | created_timestamp          | last_status      | last_status_timestamp   | created_date        |
+====+===========+===========+============================+==================+=========================+=====================+
|  0 | 332361399 |      2841 | 2025-07-14 08:38:22.250000 | proccesed_orders | 2025-07-14 11:35:22     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+
|  1 | 153141296 |      3681 | 2025-07-14 09:13:56.410000 | proccesed_orders | 2025-07-14 10:47:53     | 2025-07-14 00:00:00 |
+----+-----------+-----------+----------------------------+------------------+-------------------------+---------------------+

📊 Схема данных:
  • ite

In [17]:
# Шаг 2.5. Загрузка взаимодействий (tracker)
def load_tracker():
    print("Загружаем тренировочные данные взаимодействий...")
    tracker_data = []
    # Используем rglob для рекурсивного поиска всех .parquet файлов
    # в директории TRACKER_PATH и её поддиректориях
    for f in Path(TRACKER_PATH).rglob('*.parquet'):
        tracker_data.append(pd.read_parquet(f))

    if tracker_data:
        df = pd.concat(tracker_data, ignore_index=True)
        print(f"Загружено взаимодействий: {len(df):,}")
        return df
    else:
        print("Файлы взаимодействий не найдены.")
        return pd.DataFrame() # Возвращаем пустой DataFrame

tracker_df = load_tracker()

Загружаем тренировочные данные взаимодействий...
Загружено взаимодействий: 116,552,409


In [18]:
print("\n\n=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===")
print("\n👉 Первые 2 строки с типами данных:")
print(tracker_df.head(2).to_markdown(tablefmt="grid"))

print("\n📊 Схема данных:")
for col in tracker_df.columns:
    print(f"  • {col}: {tracker_df[col].dtype}")



=== 📙 ДАННЫЕ ВЗАИМОДЕЙСТВИЙ (tracker_df) ===

👉 Первые 2 строки с типами данных:
+----+-----------------+-----------+-----------+---------------------+---------------+
|    | action_widget   |   item_id |   user_id | timestamp           | action_type   |
+====+=================+===========+===========+=====================+===============+
|  0 | pdp             |    996252 |   3542470 | 2025-07-08 23:29:23 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+
|  1 | pdp             |   4127285 |   3267450 | 2025-07-09 14:51:02 | to_cart       |
+----+-----------------+-----------+-----------+---------------------+---------------+

📊 Схема данных:
  • action_widget: object
  • item_id: int32
  • user_id: int32
  • timestamp: datetime64[ns]
  • action_type: object


In [19]:
# Шаг 3. Загрузка тестовых пользователей
def load_test_users():
    print("Загружаем тестовых пользователей...")
    test_files = glob.glob(f'{TEST_PATH}/*.parquet')
    users = set()
    for f in tqdm(test_files, desc="Обработка тестовых файлов"):
        df_part = pd.read_parquet(f)
        if 'user_id' in df_part.columns:
            users.update(df_part['user_id'].unique())
    print(f"Найдено уникальных тестовых пользователей: {len(users):,}")
    return list(users)

test_users = load_test_users()

Загружаем тестовых пользователей...


Обработка тестовых файлов: 100%|██████████| 1/1 [00:00<00:00,  7.59it/s]

Найдено уникальных тестовых пользователей: 470,347


In [20]:
print("=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
test_users_df = pd.DataFrame({'user_id': test_users})
print(f"📊 Размер: {len(test_users_df):,} строк")
print("\n👉 Первые 2 строки:")
print(test_users_df.head(2).to_string())
print("\n📋 Колонки и типы данных:")
for col in test_users_df.columns:
    print(f"  • {col:20} {test_users_df[col].dtype}")

=== 📗 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
📊 Размер: 470,347 строк

👉 Первые 2 строки:
   user_id
0        1
1  3145730

📋 Колонки и типы данных:
  • user_id              int32


In [21]:
print("\n\nАНАЛИЗ ЗАКАЗОВ")
print("=" * 50)
print(f"Общее количество заказов: {len(orders_df):,}")
print(f"Уникальных пользователей: {orders_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {orders_df['item_id'].nunique():,}")

if 'created_date' in orders_df.columns:
    min_date = orders_df['created_date'].min().date()
    max_date = orders_df['created_date'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'last_status' in orders_df.columns:
    print("\nРаспределение статусов заказов:")
    status_counts = orders_df['last_status'].value_counts()
    status_counts_pct = orders_df['last_status'].value_counts(normalize=True) * 100
    for status, count in status_counts.items():
        pct = status_counts_pct[status]
        print(f"  {status}: {count:,} ({pct:.1f}%)")




АНАЛИЗ ЗАКАЗОВ
Общее количество заказов: 20,362,338
Уникальных пользователей: 842,254
Уникальных товаров: 4,679,218
Период данных: 2025-01-01 - 2025-07-15

Распределение статусов заказов:
  delivered_orders: 10,420,894 (51.2%)
  canceled_orders: 8,420,631 (41.4%)
  proccesed_orders: 1,520,813 (7.5%)


In [22]:
print("\n\nАНАЛИЗ ВЗАИМОДЕЙСТВИЙ")
print("=" * 50)
print(f"Общее количество взаимодействий: {len(tracker_df):,}")
print(f"Уникальных пользователей: {tracker_df['user_id'].nunique():,}")
print(f"Уникальных товаров: {tracker_df['item_id'].nunique():,}")

if 'timestamp' in tracker_df.columns:
    min_date = tracker_df['timestamp'].min().date()
    max_date = tracker_df['timestamp'].max().date()
    print(f"Период данных: {min_date} - {max_date}")

if 'action_type' in tracker_df.columns:
    print("\nРаспределение типов действий:")
    action_counts = tracker_df['action_type'].value_counts()
    action_counts_pct = tracker_df['action_type'].value_counts(normalize=True) * 100
    for action, count in action_counts.items():
        pct = action_counts_pct[action]
        print(f"  {action}: {count:,} ({pct:.1f}%)")



АНАЛИЗ ВЗАИМОДЕЙСТВИЙ
Общее количество взаимодействий: 116,552,409
Уникальных пользователей: 807,670
Уникальных товаров: 3,197,923
Период данных: 2010-01-30 - 2025-07-16

Распределение типов действий:
  page_view: 80,605,340 (69.2%)
  view_description: 17,253,961 (14.8%)
  review_view: 5,729,054 (4.9%)
  to_cart: 4,618,070 (4.0%)
  favorite: 3,455,017 (3.0%)
  remove: 3,007,598 (2.6%)
  unfavorite: 1,883,369 (1.6%)


In [23]:
# Для tracker_df по timestamp
if 'timestamp' in tracker_df.columns:
    print("\nКоличество взаимодействий по годам (на основе timestamp):")
    tracker_by_year = tracker_df['timestamp'].dt.year.value_counts().sort_index()
    for year, count in tracker_by_year.items():
        print(f"  {year}: {count:,}")


Количество взаимодействий по годам (на основе timestamp):
  2010: 1
  2024: 12
  2025: 116,552,396


In [24]:
print("\n\nАНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)")
print("=" * 50)
# Объединим item_id из orders и tracker для более полной картины
all_item_ids = set(orders_df['item_id'].unique()).union(set(tracker_df['item_id'].unique()))
print(f"Общее количество уникальных товаров в заказах и взаимодействиях: {len(all_item_ids):,}")



АНАЛИЗ ТОВАРОВ (на основе заказов и взаимодействий)
Общее количество уникальных товаров в заказах и взаимодействиях: 4,797,217


In [25]:
print("\nТоп-10 самых популярных товаров (по количеству заказов):")
top_items_orders = orders_df['item_id'].value_counts().head(10)
for item_id, count in top_items_orders.items():
    print(f"  Товар {item_id}: {count:,} заказов")



Топ-10 самых популярных товаров (по количеству заказов):
  Товар 51974017: 13,361 заказов
  Товар 187052809: 12,384 заказов
  Товар 207631139: 8,877 заказов
  Товар 143497612: 4,096 заказов
  Товар 119105606: 3,497 заказов
  Товар 247423473: 3,188 заказов
  Товар 77696741: 2,735 заказов
  Товар 175287070: 2,725 заказов
  Товар 285009143: 2,634 заказов
  Товар 201930716: 2,624 заказов


In [26]:
# Шаг 4. Разделение на тренировочную и тестовую выборки по времени
print("=== 📊 РАЗДЕЛЕНИЕ НА TRAIN/TEST ПО ВРЕМЕНИ ===")
train_end_date = pd.to_datetime('2025-07-08')  # Первая неделя: 2025-07-02 до 2025-07-08
test_start_date = pd.to_datetime('2025-07-09')  # Вторая неделя: 2025-07-09 до 2025-07-15

# Тренировочные данные (только доставленные заказы за первую неделю)
train_orders = orders_df[
    (orders_df['last_status'] == 'delivered_orders') &
    (orders_df['created_date'] >= pd.to_datetime('2025-07-02')) &
    (orders_df['created_date'] <= train_end_date)
].copy()

# Тестовые данные (заказы за вторую неделю - для оценки)
test_orders = orders_df[
    (orders_df['last_status'] == 'delivered_orders') &
    (orders_df['created_date'] >= test_start_date) &
    (orders_df['created_date'] <= pd.to_datetime('2025-07-15'))
].copy()

print(f"📅 Тренировочный период: 2025-07-02 - {train_end_date.date()}")
print(f"📅 Тестовый период: {test_start_date.date()} - 2025-07-15")
print(f"📦 Тренировочных заказов: {len(train_orders):,}")
print(f"📦 Тестовых заказов: {len(test_orders):,}")

=== 📊 РАЗДЕЛЕНИЕ НА TRAIN/TEST ПО ВРЕМЕНИ ===
📅 Тренировочный период: 2025-07-02 - 2025-07-08
📅 Тестовый период: 2025-07-09 - 2025-07-15
📦 Тренировочных заказов: 317,404
📦 Тестовых заказов: 167,241


In [27]:
# Шаг 5. Аналитическая "модель популярности" по тренировочным данным
def get_popular_items(train_orders, top_k=100):
    """Получить топ-K популярных товаров из тренировочной выборки"""
    print("\n=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===")
    item_counts = train_orders['item_id'].value_counts().head(top_k)
    popular_items = item_counts.index.tolist()

    print(f"Топ-{top_k} популярных товаров рассчитан")
    print("\n📋 Топ-10 самых популярных товаров:")
    for i, (item_id, count) in enumerate(item_counts.head(10).items(), 1):
        print(f"  {i:2d}. Товар {item_id:>12}: {count:>6,} заказов")

    return popular_items

popular_items = get_popular_items(train_orders, top_k=100)


=== 🏆 РАСЧЕТ ПОПУЛЯРНЫХ ТОВАРОВ (TRAIN) ===
Топ-100 популярных товаров рассчитан

📋 Топ-10 самых популярных товаров:
   1. Товар    175287070:    343 заказов
   2. Товар     51974017:    306 заказов
   3. Товар    166327353:    263 заказов
   4. Товар    187052809:    262 заказов
   5. Товар    247423473:    204 заказов
   6. Товар     11083343:    195 заказов
   7. Товар    334086992:    184 заказов
   8. Товар     63987378:    171 заказов
   9. Товар    206494927:    164 заказов
  10. Товар    201930716:    136 заказов


In [28]:
# Шаг 6. Получение тестовых пользователей (те, кто сделал заказы во вторую неделю)
test_users_during_period = test_orders['user_id'].unique().tolist()
print(f"\n=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===")
print(f"Найдено тестовых пользователей: {len(test_users_during_period):,}")


=== 👥 ТЕСТОВЫЕ ПОЛЬЗОВАТЕЛИ ===
Найдено тестовых пользователей: 96,159


In [29]:
# Преобразуем список тестовых пользователей в множество для быстрого поиска
test_users_set = set(test_users_during_period)
print(f"✅ Множество тестовых пользователей создано. Размер: {len(test_users_set):,}")

✅ Множество тестовых пользователей создано. Размер: 96,159


In [30]:
# --- Создание оптимизированного tracker_df с последними 50 взаимодействиями для тестовых пользователей ---

import gc
import sys

# --- 1. Убедитесь, что необходимые переменные определены ---
# train_end_date = pd.to_datetime('2025-07-08') # Пример
# test_users_set = set(test_users) # Убедитесь, что это определено

print("=== Создание оптимизированного tracker_df (последние 50 действий на пользователя) ===")

# --- 2. Фильтрация tracker_df ---
# Оставляем только взаимодействия до train_end_date для тестовых пользователей
print("Фильтрация tracker_df по дате и пользователям...")
filtered_tracker = tracker_df[
    (tracker_df['timestamp'] <= train_end_date) &
    (tracker_df['user_id'].isin(test_users_set))
].copy()

print(f"  Исходный tracker_df: {len(tracker_df):,} строк")
print(f"  Отфильтрованный tracker_df: {len(filtered_tracker):,} строк")
if len(tracker_df) > 0:
    reduction_percent = (1 - len(filtered_tracker) / len(tracker_df)) * 100
    print(f"  Сокращение строк: {reduction_percent:.1f}%")

# --- 3. Получение последних 50 взаимодействий для каждого пользователя ---
print("\nОтбор последних 50 взаимодействий для каждого пользователя...")
# Сортировка по пользователю и времени (по убыванию времени)
filtered_tracker_sorted = filtered_tracker.sort_values(['user_id', 'timestamp'], ascending=[True, False])

# Группировка по пользователю и взятие первых 50 строк (последние по времени)
tracker_df_optimized = filtered_tracker_sorted.groupby('user_id').head(50).copy()
# Явно сбрасываем индекс
tracker_df_optimized.reset_index(drop=True, inplace=True)

print(f"  tracker_df_optimized: {len(tracker_df_optimized):,} строк")
if len(filtered_tracker) > 0:
    reduction_percent_2 = (1 - len(tracker_df_optimized) / len(filtered_tracker)) * 100
    print(f"  Дополнительное сокращение строк: {reduction_percent_2:.1f}%")

# --- 4. Опционально: Оптимизация типов данных ---
print("\nОптимизация типов данных...")
# action_type и action_widget - категории, если уникальных значений немного
for col in ['action_type', 'action_widget']:
    if col in tracker_df_optimized.columns:
        if tracker_df_optimized[col].nunique() / len(tracker_df_optimized) < 0.5: # Эвристика
            tracker_df_optimized[col] = tracker_df_optimized[col].astype('category')
            print(f"  - {col} преобразован в category")

# --- 5. Удаление оригинального tracker_df и замена ---
print("\nЗамена оригинального tracker_df...")
tracker_memory_before = sys.getsizeof(tracker_df)

# Явно удаляем оригинальные большие датафреймы
del tracker_df
del filtered_tracker
del filtered_tracker_sorted
gc.collect()

# Присваиваем оптимизированный датафрейм
tracker_df = tracker_df_optimized
del tracker_df_optimized # Удаляем временную переменную

tracker_memory_after = sys.getsizeof(tracker_df)
print(f"  Память до (оценка): ~{tracker_memory_before / (1024**3):.1f} GiB")
print(f"  Память после (оценка): ~{tracker_memory_after / (1024**3):.1f} GiB")
if tracker_memory_before > 0:
    print(f"  Экономия (оценка): ~{(tracker_memory_before - tracker_memory_after) / (1024**3):.1f} GiB")

print("\n✅ Оптимизированный tracker_df готов к использованию.")
print(f"   Размер: {len(tracker_df):,} строк, {len(tracker_df.columns)} столбцов.")

# --- 6. Проверка результата ---
print("\n--- Проверка ---")
print("Несколько строк оптимизированного tracker_df:")
print(tracker_df.head())
print("\nИнформация о памяти:")
tracker_df.info(memory_usage='deep')


=== Создание оптимизированного tracker_df (последние 50 действий на пользователя) ===
Фильтрация tracker_df по дате и пользователям...
  Исходный tracker_df: 116,552,409 строк
  Отфильтрованный tracker_df: 12,635,542 строк
  Сокращение строк: 89.2%

Отбор последних 50 взаимодействий для каждого пользователя...
  tracker_df_optimized: 3,325,453 строк
  Дополнительное сокращение строк: 73.7%

Оптимизация типов данных...
  - action_type преобразован в category
  - action_widget преобразован в category

Замена оригинального tracker_df...
  Память до (оценка): ~14.3 GiB
  Память после (оценка): ~0.1 GiB
  Экономия (оценка): ~14.2 GiB

✅ Оптимизированный tracker_df готов к использованию.
   Размер: 3,325,453 строк, 5 столбцов.

--- Проверка ---
Несколько строк оптимизированного tracker_df:
   action_widget    item_id  user_id           timestamp  action_type
0        analogs  118336364       60 2025-07-07 10:33:15    page_view
1        analogs  118336364       60 2025-07-07 10:33:10    page_

In [34]:
# Шаг 7. Вычисление истории покупок для тестовых пользователей (их покупки в первую неделю)
def build_user_history_first_week(orders_df, test_users, train_end_date):
    """Построить историю покупок тестовых пользователей до тестового периода"""
    print("\n=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (до теста) ===")

    # Все доставленные заказы тестовых пользователей до начала теста
    user_history = orders_df[
        (orders_df['last_status'] == 'delivered_orders') &
        (orders_df['user_id'].isin(test_users)) &
        (orders_df['created_date'] <= train_end_date)
    ].groupby('user_id')['item_id'].apply(set).to_dict()

    # Конвертируем в список для совместимости
    user_preferences = {uid: list(items) for uid, items in user_history.items()}

    print(f"История покупок построена для {len(user_preferences):,} пользователей")
    return user_preferences

user_preferences = build_user_history_first_week(orders_df, test_users_during_period, train_end_date)


=== 📚 ПОСТРОЕНИЕ ИСТОРИИ ПОКУПОК (до теста) ===
История покупок построена для 94,024 пользователей


In [35]:
# Шаг 8. Генерация рекомендаций
def generate_recommendations(test_users, popular_items, user_preferences, top_k=100):
    """Генерация рекомендаций: популярные товары, исключая уже купленные"""
    print("\n=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===")
    recs = {}
    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        bought = set(user_preferences.get(uid, []))
        recs[uid] = [item for item in popular_items if item not in bought][:top_k]
    print(f"Рекомендации сгенерированы для {len(recs):,} пользователей")
    return recs

recommendations = generate_recommendations(test_users_during_period, popular_items, user_preferences)


=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ ===


Генерация рекомендаций: 100%|██████████| 96159/96159 [00:01<00:00, 48696.72it/s]


Рекомендации сгенерированы для 96,159 пользователей


In [36]:
# Шаг 10. Подготовка тестовых меток (что пользователи действительно купили во вторую неделю)
def prepare_ground_truth(test_orders):
    """Подготовить ground truth: что пользователи купили в тестовый период"""
    print("\n=== ✅ ПОДГОТОВКА GROUND TRUTH ===")
    ground_truth = test_orders.groupby('user_id')['item_id'].apply(set).to_dict()

    print(f"Ground truth подготовлен для {len(ground_truth):,} пользователей")

    # Показываем примеры для проверки
    if ground_truth:
        # Берем первых 3 пользователя для примера
        sample_users = sorted(list(ground_truth.keys()))[:3]  # Сортируем для воспроизводимости
        print("\n🔍 Примеры ground truth для проверки:")
        for i, user_id in enumerate(sample_users, 1):
            items = list(ground_truth[user_id])  # Все товары пользователя
            print(f"  Пользователь {user_id}: {items} ({len(ground_truth[user_id])} всего покупок)")

            # Дополнительная проверка - показываем те же данные из исходного тестового датафрейма
            user_test_data = test_orders[test_orders['user_id'] == user_id]
            test_items = user_test_data['item_id'].tolist()
            print(f"  Пользователь {user_id} (из теста): {test_items} ({len(test_items)} записей в тесте)")

    return ground_truth

ground_truth = prepare_ground_truth(test_orders)


=== ✅ ПОДГОТОВКА GROUND TRUTH ===
Ground truth подготовлен для 96,159 пользователей

🔍 Примеры ground truth для проверки:
  Пользователь 60: [18766015, 111169159] (2 всего покупок)
  Пользователь 60 (из теста): [111169159, 18766015] (2 записей в тесте)
  Пользователь 91: [27861200, 15428515, 27733496, 104094923] (4 всего покупок)
  Пользователь 91 (из теста): [15428515, 27861200, 27733496, 104094923] (4 записей в тесте)
  Пользователь 111: [179543784, 83982210] (2 всего покупок)
  Пользователь 111 (из теста): [179543784, 83982210] (2 записей в тесте)


In [37]:
# Шаг 10. Расчет NDCG@100
def ndcg_at_k(y_true, y_pred, k=100):
    """
    Рассчитать NDCG@k для одного пользователя
    y_true: множество реально купленных товаров
    y_pred: список рекомендованных товаров
    """
    if not y_true:
        return 0.0

    # Бинарная оценка: 1 если товар куплен, 0 если нет
    dcg = 0.0
    for i, item in enumerate(y_pred[:k]):
        if item in y_true:
            dcg += 1.0 / np.log2(i + 2)  # log2(1+pos) = log2(pos+2) т.к. индекс с 0

    # IDCG - идеальный DCG (все релевантные товары в начале)
    idcg = 0.0
    ideal_len = min(len(y_true), k)
    for i in range(ideal_len):
        idcg += 1.0 / np.log2(i + 2)

    if idcg == 0:
        return 0.0

    return dcg / idcg

def calculate_ndcg_batch(recommendations, ground_truth, k=100):
    """Рассчитать NDCG@k для всех пользователей"""
    print(f"\n=== 📊 РАСЧЕТ МЕТРИКИ NDCG@{k} ===")

    ndcg_scores = []
    users_evaluated = 0

    for uid in tqdm(recommendations.keys(), desc='Расчет NDCG'):
        if uid in ground_truth:
            y_pred = recommendations[uid]
            y_true = ground_truth[uid]

            ndcg = ndcg_at_k(y_true, y_pred, k)
            ndcg_scores.append(ndcg)
            users_evaluated += 1

    mean_ndcg = np.mean(ndcg_scores) if ndcg_scores else 0.0
    print(f"Оценено пользователей: {users_evaluated:,}")
    print(f"NDCG@{k}: {mean_ndcg:.6f}")

    return mean_ndcg

# Расчет метрики
ndcg_score = calculate_ndcg_batch(recommendations, ground_truth, k=100)


=== 📊 РАСЧЕТ МЕТРИКИ NDCG@100 ===


Расчет NDCG: 100%|██████████| 96159/96159 [00:01<00:00, 89646.21it/s]

Оценено пользователей: 96,159
NDCG@100: 0.007415


In [19]:
# БЛОК 2: Подход организаторов (только просмотры с ограничением 50)

# --- Функции для подхода организаторов ---
def get_popular_items_from_views(tracker_df, train_end_date, top_k=100):
    """Получить топ-K популярных товаров на основе просмотров (page_view)"""
    print("\n=== 📈 ПОПУЛЯРНОСТЬ НА ОСНОВЕ ПРОСМОТРОВ (метод организаторов) ===")

    # Фильтруем только просмотры до конца тренировочного периода
    view_interactions = tracker_df[
        (tracker_df['timestamp'] <= train_end_date) &
        (tracker_df['action_type'] == 'page_view')
    ]

    print(f"Анализируем просмотры: {len(view_interactions):,} записей")

    # Считаем популярность по просмотрам
    item_views = view_interactions['item_id'].value_counts()
    popular_items = item_views.head(top_k).index.tolist()

    print(f"Топ-{top_k} популярных товаров по просмотрам рассчитан")
    print("\n📋 Топ-10 самых просматриваемых товаров:")
    for i, (item_id, count) in enumerate(item_views.head(10).items(), 1):
        print(f"  {i:2d}. Товар {item_id:>12}: {count:>8,} просмотров")

    return popular_items

def build_user_viewed_history(tracker_df, test_users, train_end_date):
    """
    Построить историю просмотров для тестовых пользователей
    Используем deque maxlen=50 для каждого пользователя
    """
    print("\n=== 👁️ ИСТОРИЯ ПРОСМОТРОВ ПОЛЬЗОВАТЕЛЕЙ (ограничена 50 последними) ===")

    # Фильтруем просмотры до конца тренировочного периода для тестовых пользователей
    user_views = tracker_df[
        (tracker_df['timestamp'] <= train_end_date) &
        (tracker_df['user_id'].isin(test_users)) &
        (tracker_df['action_type'] == 'page_view')
    ]

    print(f"Анализируем пользовательские просмотры: {len(user_views):,} записей")

    # Используем словарь с deque maxlen=50 для каждого пользователя
    user_viewed_items_deque = defaultdict(lambda: deque(maxlen=50))

    # Заполняем историю просмотров (в порядке временной последовательности)
    user_views_sorted = user_views.sort_values('timestamp')

    for _, row in tqdm(user_views_sorted.iterrows(), total=len(user_views_sorted), desc="Обработка просмотров"):
        user_id = row['user_id']
        item_id = row['item_id']
        user_viewed_items_deque[user_id].append(item_id)

    # Конвертируем deque в списки для совместимости
    user_preferences = {uid: list(items_deque) for uid, items_deque in user_viewed_items_deque.items()}

    print(f"История просмотров построена для {len(user_preferences):,} пользователей")
    avg_len = np.mean([len(items) for items in user_preferences.values()]) if user_preferences else 0
    print(f"Средняя длина истории просмотров: {avg_len:.1f}")

    return user_preferences

def generate_recommendations_exclude_viewed(test_users, popular_items, user_viewed_preferences, top_k=100):
    """
    Генерация рекомендаций: популярные товары, исключая просмотренные
    """
    print("\n=== 🎯 ГЕНЕРАЦИЯ РЕКОМЕНДАЦИЙ (исключая просмотренные) ===")
    recs = {}

    for uid in tqdm(test_users, desc='Генерация рекомендаций'):
        # Получаем множество уже просмотренных (ограничено maxlen=50)
        viewed = set(user_viewed_preferences.get(uid, []))
        # Фильтруем популярные товары, исключая уже просмотренные
        recs[uid] = [item for item in popular_items if item not in viewed][:top_k]

    print(f"Рекомендации сгенерированы для {len(recs):,} пользователей")
    return recs

# --- Выполнение подхода организаторов ---
# 1. Популярность по просмотрам
view_based_popular_items = get_popular_items_from_views(tracker_df, train_end_date, top_k=100)

# 2. История просмотров пользователей
user_viewed_preferences = build_user_viewed_history(tracker_df, test_users_during_period, train_end_date)

# 3. Генерация рекомендаций с исключением просмотренных
view_based_recommendations = generate_recommendations_exclude_viewed(
    test_users_during_period, view_based_popular_items, user_viewed_preferences
)

# 4. Расчет метрики для подхода организаторов
view_based_ndcg_score = calculate_ndcg_batch(view_based_recommendations, ground_truth, k=100)
print(f"\n📊 Подход организаторов (просмотры): NDCG@100 = {view_based_ndcg_score:.6f}")

# Если базовая метрика не была рассчитана ранее
if ndcg_score_baseline is None:
    print("(Базовая метрика не была рассчитана ранее)")
else:
    improvement_views = ((view_based_ndcg_score/ndcg_score_baseline - 1)*100)
    print(f"📈 Улучшение vs базовой (просмотры): {improvement_views:+.2f}%")



=== 📈 ПОПУЛЯРНОСТЬ НА ОСНОВЕ ПРОСМОТРОВ (метод организаторов) ===
Анализируем просмотры: 34,650,068 записей
Топ-100 популярных товаров по просмотрам рассчитан

📋 Топ-10 самых просматриваемых товаров:
   1. Товар    219236221:    8,028 просмотров
   2. Товар    207631139:    7,450 просмотров
   3. Товар     12904245:    7,321 просмотров
   4. Товар    172018601:    6,280 просмотров
   5. Товар     49181059:    6,068 просмотров
   6. Товар    221521980:    5,614 просмотров
   7. Товар    214054774:    5,327 просмотров
   8. Товар    113070693:    5,290 просмотров
   9. Товар     26556597:    4,940 просмотров
  10. Товар    156941538:    4,936 просмотров

=== 👁️ ИСТОРИЯ ПРОСМОТРОВ ПОЛЬЗОВАТЕЛЕЙ (ограничена 50 последними) ===
Анализируем пользовательские просмотры: 8,727,268 записей


Обработка просмотров:   0%|          | 0/8727268 [00:10<?, ?it/s]


NameError: name 'deque' is not defined

In [ ]:
# БЛОК 3: Все взаимодействия с ограничением 50

def build_user_all_interactions_history_limited(tracker_df, test_users, train_end_date):
    """
    Построить историю ВСЕХ взаимодействий для тестовых пользователей
    Используем deque maxlen=50 для ограничения истории
    """
    print("\n=== 🔄 ИСТОРИЯ ВСЕХ ВЗАИМОДЕЙСТВИЙ (ограничена 50 последними) ===")

    # Фильтруем ВСЕ взаимодействия до конца тренировочного периода для тестовых пользователей
    user_all_interactions = tracker_df[
        (tracker_df['timestamp'] <= train_end_date) &
        (tracker_df['user_id'].isin(test_users))
    ]

    print(f"Анализируем все пользовательские взаимодействия: {len(user_all_interactions):,} записей")

    # Используем словарь с deque maxlen=50 для каждого пользователя
    user_interaction_items_deque = defaultdict(lambda: deque(maxlen=50))

    # Заполняем историю взаимодействий (в порядке временной последовательности)
    user_interactions_sorted = user_all_interactions.sort_values('timestamp')

    for _, row in tqdm(user_interactions_sorted.iterrows(), total=len(user_interactions_sorted), desc="Обработка взаимодействий"):
        user_id = row['user_id']
        item_id = row['item_id']
        user_interaction_items_deque[user_id].append(item_id)

    # Конвертируем deque в списки для совместимости
    user_preferences = {uid: list(items_deque) for uid, items_deque in user_interaction_items_deque.items()}

    print(f"История всех взаимодействий построена для {len(user_preferences):,} пользователей")
    avg_len = np.mean([len(items) for items in user_preferences.values()]) if user_preferences else 0
    print(f"Средняя длина истории взаимодействий: {avg_len:.1f}")

    return user_preferences

# --- Выполнение подхода "все взаимодействия с ограничением 50" ---
# Используем уже рассчитанную ранее популярность по заказам или просмотрам
# Для чистоты эксперимента, можно использовать ту же популярность, что и у организаторов,
# или пересчитать на основе всех действий. Пока используем view_based_popular_items.

# 1. История всех взаимодействий с ограничением 50
all_interactions_limited_preferences = build_user_all_interactions_history_limited(
    tracker_df, test_users_during_period, train_end_date
)

# 2. Генерация рекомендаций (используем ту же функцию, что и для просмотров)
all_interactions_limited_recommendations = generate_recommendations_exclude_viewed(
    test_users_during_period, view_based_popular_items, all_interactions_limited_preferences # Или используйте другую популярность
)

# 3. Расчет метрики
all_interactions_limited_ndcg_score = calculate_ndcg_batch(all_interactions_limited_recommendations, ground_truth, k=100)
print(f"\n📊 Все взаимодействия с ограничением 50: NDCG@100 = {all_interactions_limited_ndcg_score:.6f}")

# Сравнение с подходом организаторов
improvement_all_limited = ((all_interactions_limited_ndcg_score/view_based_ndcg_score - 1)*100)
print(f"📈 Улучшение vs подхода организаторов (все действия, лимит 50): {improvement_all_limited:+.2f}%")

# Если базовая метрика была рассчитана
if ndcg_score_baseline is not None:
    improvement_all_limited_vs_baseline = ((all_interactions_limited_ndcg_score/ndcg_score_baseline - 1)*100)
    print(f"📈 Улучшение vs базовой (все действия, лимит 50): {improvement_all_limited_vs_baseline:+.2f}%")
